# Model Reverse Engineering

In [6]:
N_SAMPLES = 100

MIN_FLIPPER_LENGTH = 150
MAX_FLIPPER_LENGTH = 250

MIN_BODY_MASS = 2500
MAX_BODY_MASS = 6500

CLASSIFIER_URL = "http://154.57.164.73:32223/"


In [4]:
import random
import pandas as pd

samples = {
    "Flipper Length (mm)": [],
    "Body Mass (g)": []
}

for i in range(N_SAMPLES):
    samples["Flipper Length (mm)"].append(random.uniform(MIN_FLIPPER_LENGTH, MAX_FLIPPER_LENGTH))
    samples["Body Mass (g)"].append(random.uniform(MIN_BODY_MASS, MAX_BODY_MASS))

samples_df = pd.DataFrame(samples)
print(samples_df.head())

   Flipper Length (mm)  Body Mass (g)
0           227.425804    2963.810966
1           192.114844    2589.119814
2           189.162029    3867.875186
3           235.267525    5478.570456
4           161.730026    4001.410718


In [7]:
import requests
import json

predictions = {"species": []}

for i in range(N_SAMPLES):
    sample = {
                "flipper_length": samples["Flipper Length (mm)"][i],
                "body_mass": samples["Body Mass (g)"][i]
            }

    prediction = json.loads(requests.get(CLASSIFIER_URL, params=sample).text).get("result")
    predictions["species"].append(prediction)

predictions_df = pd.DataFrame(predictions)
print(predictions_df.head())


  species
0  Gentoo
1  Adelie
2  Adelie
3  Gentoo
4  Adelie


In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

surrogate_model = make_pipeline(StandardScaler(), LogisticRegression())
surrogate_model.fit(samples_df, predictions_df)

# save classifier to a file
joblib.dump(surrogate_model, 'surrogate.joblib')

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['surrogate.joblib']

In [9]:
with open('surrogate.joblib', 'rb') as f:
    file = f.read()

r = requests.post(CLASSIFIER_URL + '/model', files={'file': ('surrogate.joblib', file)})

print(json.loads(r.text))

{'accuracy': 0.9854014598540146, 'flag': 'HTB{ff08c0bb37e16f30a0804053a4de70ed}'}


# Denial of ML Service

## Sponge Examples

Sponge Examples, as introduced in this paper [https://arxiv.org/pdf/2006.03463], are specifically crafted adversarial inputs that maximize energy consumption and latency in the ML mode. As such, inference on these inputs forces the model to perform in the worst possible way in terms of power consumption and latency, making them a perfect fit for DoS attacks. In cases where the ML deployment runs on a battery-powered device (e.g., smartphones), high energy consumption may even drain the battery entirely, causing a complete outage in the ML system.

The basic idea behind sponge examples is to create ML inputs that result in high energy consumption or inference latency without increasing the input dimension. For obvious reasons, higher-dimensional inputs will require more computational effort to process. For instance, a longer input prompt to an LLM or a higher resolution input image to a visual model will require more computational work to process, thus directly resulting in higher energy consumption and latency. However, limiting the input dimension can prevent DoS attacks resulting from overly high-dimensional inputs. For instance, the input prompt to language models can be cut off after a maximum length, an input image scaled down to a lower resolution, or rejected entirely if it is too large. Therefore, sponge examples aim to increase computational work without increasing the input dimension.

----

Creating Sponge Examples
We can use a white-box or a black-box approach to create sponge examples. For the white-box approach, we need to be able to access the model's architectures and parameters. These requirements are unlikely to be met in real-world AI deployments. However, adversaries can use this approach by hosting their own classifier and developing sponge examples locally, which may transfer to other AI deployments using a similar architecture.

On the other hand, the black-box approach only requires adversaries to be able to query the model and subsequently measure either energy consumption or inference latency. While measuring the model's energy consumption is typically impossible, measuring inference latency is often realistic in classification services. Adversaries can simply query the model and measure the time it takes to compute the classification result to deduce the inference latency. Overall, the requirements for the black-box approach are commonly met in many real-world AI deployments.

# Insecure Integrated Components

Real-world ML applications often comprise a vast array of interacting components. The entire ML application may be at risk if any of these suffer from security vulnerabilities. Common examples of insecure integrated components include a web application in which an ML model is integrated. Security vulnerabilities in the web application may put ML-related data at risk. Another example is a plugin supported by the ML model. Plugins are extensions that complex ML models, such as LLMs, can dynamically invoke based on the user's query to provide additional functionality. This can include querying databases, calling external APIs, or retrieving real-time information from external sources. Security vulnerabilities resulting from insecure integrated components can affect both the model input and output, depending on the type of vulnerability.

# Rogue Actions

In ML applications, rogue actions refer to unintended behaviors or operations carried out via system extensions, such as LLM plugins or agents. These actions may arise accidentally due to poor alignment between the model input and the system's constraints, or malicious adversaries may trigger them intentionally by exploiting vulnerabilities or using prompt injection. As such, it can be challenging to determine if a rogue action was caused by the inherent randomness of ML models or an adversary's malicious input. Due to the increasing modularity and extensibility of ML applications, agents are often granted access to plugins or extensions to enhance their functionality. These can include APIs, automation routines, or third-party integrations. While these extensions are intended to expand the agent's capabilities, they also significantly increase the attack surface. If an agent is not properly sandboxed or its actions are not adequately restricted, it might execute harmful commands that impact data integrity, privacy, or system operations.

Due to AI's inherent randomness, all AI applications are at risk of rogue actions if the model behaves unexpectedly. For instance, in July 2025, Replit's AI coding tool executed a rogue action when it deleted a production database, violating explicit constraints it was given. However, this section will focus on rogue actions that adversaries may provoke maliciously through prompt injection.



# MCP Server Vulnerabilities